# LogQ on BookCrossing — CDN-style baseline

## 0. Config and imports

In [1]:
import gc
import itertools
import json
import re
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


def set_seed(seed: int = 0):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _check_device():
    """Runtime CUDA check: falls back to CPU if GPU kernels are incompatible."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        torch.zeros(1, device="cuda")
        return torch.device("cuda")
    except RuntimeError:
        print("CUDA available but kernel not compatible with this GPU — using CPU.")
        return torch.device("cpu")


device = _check_device()

CFG = {
    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------
    "work_dir": "../data",
    "min_user_interactions": 3,

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------
    "embed_dim": 64,
    "tower_hidden": [256, 128, 64],
    "max_title_words": 2000,
    "max_authors": 200,
    "max_publishers": 200,
    "max_countries": 50,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    "batch_size": 1024,
    "num_workers": 0,
    "grad_clip": 1.0,

    # --------------------------------------------------------
    # Grid search
    # --------------------------------------------------------
    "tune_fast": True,
    "tune_epochs": 3,
    "tune_val_fraction": 1.0,
    "skip_tune_if_cached": True,
    "cache_path": "bookcrossing_logq_best_hparams.json",

    # --------------------------------------------------------
    # Final training
    # --------------------------------------------------------
    "final_epochs": 40,
    "seeds": [0, 1, 2, 3, 4],

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 1024,
    "head_fraction": 0.001,

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "seed": 0,
}

set_seed(CFG["seed"])
print("device:", device)
print(f"Final: {len(CFG['seeds'])} seeds × {CFG['final_epochs']} epochs")

device: cuda
Final: 5 seeds × 40 epochs


## 1. Load BookCrossing

In [2]:
def read_bookcrossing(raw: Path):
    users = pd.read_csv(
        raw / "BX-Users.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
    )

    books = pd.read_csv(
        raw / "BX-Books.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
        on_bad_lines="skip",
        low_memory=False,
    )
    print()
    books = books.drop(columns=[c for c in books.columns if "Image-URL" in c], errors="ignore")
    year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
    books["Year-Of-Publication"] = year.where(year.between(1800, 2024), 0).fillna(0).astype(int)

    ratings = pd.read_csv(
        raw / "BX-Book-Ratings.csv",
        sep=";",
        encoding="latin-1",
        quotechar='"',
    )
    ratings = ratings[ratings["Book-Rating"] > 0].copy()

    return users, books, ratings


RAW_DIR = Path(CFG["work_dir"])
users_df, books_df, ratings = read_bookcrossing(RAW_DIR)

valid_isbns = set(books_df["ISBN"].tolist())
valid_uids = set(users_df["User-ID"].tolist())
ratings = ratings[ratings["ISBN"].isin(valid_isbns) & ratings["User-ID"].isin(valid_uids)].copy()

user_counts = ratings.groupby("User-ID").size()
active_users = user_counts[user_counts >= CFG["min_user_interactions"]].index
ratings = ratings[ratings["User-ID"].isin(active_users)].copy()

all_user_ids = sorted(ratings["User-ID"].unique())
all_item_ids = sorted(ratings["ISBN"].unique())
uid_map = {u: i for i, u in enumerate(all_user_ids)}
iid_map = {b: i for i, b in enumerate(all_item_ids)}

NUM_USERS = len(uid_map)
NUM_ITEMS = len(iid_map)

print(f"users={NUM_USERS:,} items={NUM_ITEMS:,} ratings={len(ratings):,}")


users=20,194 items=137,409 ratings=327,271


## 2. Feature engineering

In [3]:
def tokenize_title(title: str):
    if "(" in title:
        title = title.rsplit("(", 1)[0]
    return re.findall(r"[a-z0-9]+", title.lower())


def build_title_bow(titles: list, max_words: int) -> np.ndarray:
    counter = Counter()
    per_title = []
    for title in titles:
        toks = tokenize_title(str(title))
        per_title.append(toks)
        counter.update(toks)
    vocab = [w for w, _ in counter.most_common(max_words)]
    w2i = {w: i for i, w in enumerate(vocab)}
    mat = np.zeros((len(titles), len(vocab)), dtype=np.float32)
    for r, toks in enumerate(per_title):
        for tok in toks:
            j = w2i.get(tok)
            if j is not None:
                mat[r, j] = 1.0
    return mat


isbn_to_title = books_df.set_index("ISBN")["Book-Title"].to_dict()
isbn_to_author = books_df.set_index("ISBN")["Book-Author"].to_dict()
isbn_to_year = books_df.set_index("ISBN")["Year-Of-Publication"].to_dict()
isbn_to_publisher = books_df.set_index("ISBN")["Publisher"].to_dict()

# ---------- item features ----------
titles = [str(isbn_to_title.get(isbn, "")) for isbn in all_item_ids]
authors = [str(isbn_to_author.get(isbn, "unknown")) for isbn in all_item_ids]
publishers = [str(isbn_to_publisher.get(isbn, "unknown")) for isbn in all_item_ids]
years_raw = np.array([float(isbn_to_year.get(isbn, 0)) for isbn in all_item_ids], dtype=np.float32)

title_bow = build_title_bow(titles, CFG["max_title_words"])

author_counter = Counter(authors)
top_author_map = {a: i for i, (a, _) in enumerate(author_counter.most_common(CFG["max_authors"]))}
author_mat = np.zeros((NUM_ITEMS, len(top_author_map)), dtype=np.float32)
for r, a in enumerate(authors):
    j = top_author_map.get(a)
    if j is not None:
        author_mat[r, j] = 1.0

publisher_counter = Counter(publishers)
top_publisher_map = {p: i for i, (p, _) in enumerate(publisher_counter.most_common(CFG["max_publishers"]))}
publisher_mat = np.zeros((NUM_ITEMS, len(top_publisher_map)), dtype=np.float32)
for r, p in enumerate(publishers):
    j = top_publisher_map.get(p)
    if j is not None:
        publisher_mat[r, j] = 1.0

years_norm = (years_raw - years_raw.mean()) / (years_raw.std() + 1e-6)

ITEM_GEN = np.hstack([
    author_mat,
    publisher_mat,
    years_norm.reshape(-1, 1),
    title_bow,
]).astype(np.float32)
ITEM_GEN = (ITEM_GEN - ITEM_GEN.mean(axis=0, keepdims=True)) / (ITEM_GEN.std(axis=0, keepdims=True) + 1e-6)
ITEM_GEN = ITEM_GEN.astype(np.float32)
ITEM_GEN_DIM = ITEM_GEN.shape[1]

# ---------- user features ----------
def age_bucket(age) -> int:
    """0=unknown, 1=<18, 2=18-24, 3=25-34, 4=35-44, 5=45-54, 6=55+"""
    try:
        a = float(age)
    except (TypeError, ValueError):
        return 0
    if a < 18:   return 1
    if a < 25:   return 2
    if a < 35:   return 3
    if a < 45:   return 4
    if a < 55:   return 5
    return 6


def extract_country(loc) -> str:
    if pd.isna(loc) or str(loc).strip() == "":
        return "unknown"
    parts = [p.strip() for p in str(loc).split(",")]
    return parts[-1].lower().strip() or "unknown"


users_df["age_bucket"] = users_df["Age"].apply(age_bucket)
users_df["country"] = users_df["Location"].apply(extract_country)

top_countries = set(users_df["country"].value_counts().head(CFG["max_countries"]).index)
users_df["country_key"] = users_df["country"].apply(lambda c: c if c in top_countries else "other")
country_map = {c: i for i, c in enumerate(sorted(users_df["country_key"].unique()))}

user_age_arr = np.zeros(NUM_USERS, dtype=np.int64)
user_country_arr = np.zeros(NUM_USERS, dtype=np.int64)
uid_set = set(uid_map.keys())
for _, row in users_df[users_df["User-ID"].isin(uid_set)].iterrows():
    u = uid_map[row["User-ID"]]
    user_age_arr[u] = row["age_bucket"]
    user_country_arr[u] = country_map.get(row["country_key"], 0)

USER_GEN = np.stack([user_age_arr, user_country_arr], axis=1).astype(np.int64)
USER_GEN_SIZES = [7, len(country_map)]

print(f"ITEM_GEN_DIM: {ITEM_GEN_DIM}")
print(f"USER_GEN_SIZES: {USER_GEN_SIZES}")

ITEM_GEN_DIM: 2401
USER_GEN_SIZES: [7, 51]


## 3. Leave-one-out split

In [4]:
train_pairs = []
val_u, val_i = [], []
test_u, test_i = [], []

train_user_items = [set() for _ in range(NUM_USERS)]

for uid, grp in ratings.groupby("User-ID"):
    if uid not in uid_map:
        continue
    u = uid_map[uid]
    # No timestamp in BookCrossing: sort by ISBN string for a deterministic split
    item_seq = [iid_map[b] for b in sorted(grp["ISBN"].tolist()) if b in iid_map]
    if len(item_seq) < 3:
        continue
    for it in item_seq[:-2]:
        train_pairs.append((u, it))
        train_user_items[u].add(it)
    val_u.append(u)
    val_i.append(item_seq[-2])
    test_u.append(u)
    test_i.append(item_seq[-1])

train_pairs = np.asarray(train_pairs, dtype=np.int64)
val_u = np.asarray(val_u, dtype=np.int64)
val_i = np.asarray(val_i, dtype=np.int64)
test_u = np.asarray(test_u, dtype=np.int64)
test_i = np.asarray(test_i, dtype=np.int64)

print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")

known_val = [set(s) for s in train_user_items]

known_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)

order = np.argsort(-item_freq)
n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"test_head_share={head_mask[test_i].mean():.4f}")
print(f"test_tail_share={(~head_mask[test_i]).mean():.4f}")

train=286,883 val=20,194 test=20,194
head=137 tail=137,272
test_head_share=0.0296
test_tail_share=0.9704


## 4. Model

In [5]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UserGenEnc(nn.Module):
    def __init__(self, sizes: list[int], dim: int = 16):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, dim) for n in sizes])
        self.out_dim = len(sizes) * dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([emb(x[:, i].long()) for i, emb in enumerate(self.embs)], dim=-1)


class TwoTower(nn.Module):
    def __init__(self, dropout: float = 0.0):
        super().__init__()
        h = CFG["tower_hidden"]
        ed = CFG["embed_dim"]

        self.user_id = nn.Embedding(NUM_USERS, ed)
        self.user_gen = UserGenEnc(USER_GEN_SIZES, dim=16)
        self.user_mlp = MLP(ed + self.user_gen.out_dim, h, dropout)
        self.user_ln = nn.LayerNorm(self.user_mlp.out_dim)

        self.item_id = nn.Embedding(NUM_ITEMS, ed)
        self.item_mlp = MLP(ed + ITEM_GEN_DIM, h, dropout)
        self.item_ln = nn.LayerNorm(self.item_mlp.out_dim)

        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_vec(self, u: torch.Tensor, ug: torch.Tensor) -> torch.Tensor:
        x = torch.cat([self.user_id(u), self.user_gen(ug)], dim=-1)
        return self.user_ln(self.user_mlp(x))

    def item_vec(self, i: torch.Tensor, ig: torch.Tensor) -> torch.Tensor:
        x = torch.cat([self.item_id(i), ig], dim=-1)
        return self.item_ln(self.item_mlp(x))


ug_t = torch.from_numpy(USER_GEN).long().to(device)
ig_t = torch.from_numpy(ITEM_GEN).float().to(device)

train_loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=CFG["num_workers"],
    pin_memory=(device.type == "cuda"),
)

print("Model helpers ready.")
print(f"train batches per epoch: {len(train_loader)}")

Model helpers ready.
train batches per epoch: 280


## 5. Evaluation

In [ ]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 1024,
):
    """Full-catalog evaluation with ranking/topK kept on torch device."""
    model.eval()
    ranks_all, ranks_head, ranks_tail = [], [], []
    max_k = max(ks)
    recommended_by_k = {k: [] for k in ks}

    head_mask_t = torch.from_numpy(head_mask).to(device=device, dtype=torch.bool)
    tail_mask_np = ~head_mask
    popularity = np.log1p(item_freq.astype(np.float64))

    item_vectors = []
    for s in tqdm(range(0, NUM_ITEMS, item_batch_size), desc="item vectors", leave=False):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        v = model.item_vec(idx, ig_t[idx])
        v = F.normalize(v, dim=-1, eps=1e-6)
        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on batch {s}:{e}")
        item_vectors.append(v)
    item_vectors = torch.cat(item_vectors, dim=0)

    arange_cache = {}
    for start in tqdm(range(0, len(users), user_batch_size), desc="eval users", leave=False):
        end = min(start + user_batch_size, len(users))
        bu = users[start:end]
        bi = true_items[start:end]
        batch_size = len(bu)

        ut = torch.tensor(bu, device=device, dtype=torch.long)
        true_t = torch.tensor(bi, device=device, dtype=torch.long)
        uvec = model.user_vec(ut, ug_t[ut])
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)
        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on batch {start}:{end}")

        scores = uvec @ item_vectors.T
        if not torch.isfinite(scores).all():
            bad = int((~torch.isfinite(scores)).sum().item())
            raise RuntimeError(f"Non-finite scores in user batch {start}:{end}: {bad}")

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            known = [it for it in known_user_items[int(u)] if it != int(true_i)]
            if known:
                scores[row, torch.tensor(known, device=device, dtype=torch.long)] = -1e9

        if batch_size not in arange_cache:
            arange_cache[batch_size] = torch.arange(batch_size, device=device)
        rows = arange_cache[batch_size]
        true_scores = scores[rows, true_t]
        eps = 1e-12
        ranks_t = (scores > true_scores[:, None] + eps).sum(dim=1) + (torch.abs(scores - true_scores[:, None]) <= eps).sum(dim=1) - 1
        ranks = ranks_t.detach().cpu().numpy().astype(np.int64)
        true_is_head = head_mask_t[true_t].detach().cpu().numpy().astype(bool)

        ranks_all.extend(ranks.tolist())
        ranks_head.extend(ranks[true_is_head].tolist())
        ranks_tail.extend(ranks[~true_is_head].tolist())

        top_idx = torch.topk(scores, k=min(max_k, NUM_ITEMS), dim=1).indices.detach().cpu().numpy().astype(np.int64)
        for k in ks:
            recommended_by_k[k].append(top_idx[:, :k])

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}
        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k
                out[f"HR@{k}"] = 100.0 * hits.mean()
                out[f"NDCG@{k}"] = 100.0 * np.where(hits, 1.0 / np.log2(a + 2), 0.0).mean()
        return out

    num_tail_items = int(tail_mask_np.sum())
    long_tail = {}
    for k in ks:
        recs = np.vstack(recommended_by_k[k])
        unique_recs = np.unique(recs)
        catalog_coverage = len(unique_recs) / NUM_ITEMS
        tail_coverage = np.sum(tail_mask_np[unique_recs]) / num_tail_items if num_tail_items > 0 else np.nan
        avg_popularity = popularity[recs].mean()
        tail_ratio = tail_mask_np[recs].mean()
        n_users_eval = recs.shape[0]
        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(recs.reshape(-1), minlength=NUM_ITEMS)
            pairwise_intersections = np.sum(exposure_counts * (exposure_counts - 1) / 2)
            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs
            personalization = 1.0 - avg_overlap / k
        long_tail[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail,
        "counts": {"overall": len(ranks_all), "head": len(ranks_head), "tail": len(ranks_tail)},
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))
    for split in ("overall", "head", "tail"):
        print(f"[{split}]", metrics[split])
    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])


## 6. Method-specific loss (LogQ)

In [7]:
# q(i) = sampling probability of item i in in-batch negatives = freq(i) / sum(freq)
# LogQ corrects for sampling bias: logits -= logq_weight * log(q(i))
# weight=1.0 is the unbiased correction; tuning it controls the strength.
item_freq_float = item_freq.astype(np.float64)
item_freq_nonzero = np.maximum(item_freq_float, 1.0)
item_probs = item_freq_nonzero / item_freq_nonzero.sum()
item_logq = torch.from_numpy(
    np.log(np.maximum(item_probs, 1e-12)).astype(np.float32)
).to(device)

print("item_logq ready:", item_logq.shape)


def method_loss(
    user_vecs: torch.Tensor,
    item_vecs: torch.Tensor,
    items: torch.Tensor,
    logq_weight: float,
) -> torch.Tensor:
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)
    logits = u @ v.T
    # Subtract log-sampling-probability of in-batch items (column-wise correction)
    logits = logits - logq_weight * item_logq[items].unsqueeze(0)
    logits = torch.clamp(logits, -20.0, 20.0)
    if not torch.isfinite(logits).all():
        raise RuntimeError("NaN/Inf in logits after LogQ correction")
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)

item_logq ready: torch.Size([137409])


## 7. Grid search

In [8]:
# LR, dropout, weight_decay are fixed as specified.
# LOGQ_WEIGHT controls the strength of the sampling-bias correction:
#   0.1  → light correction (closer to plain in-batch softmax)
#   1.0  → theoretically unbiased correction
#   10.0 → aggressive over-correction (penalises popular items heavily)
LR_GRID          = [0.001]
DROPOUT_GRID     = [0.1]
WD_GRID          = [0.0]
LOGQ_WEIGHT_GRID = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID, LOGQ_WEIGHT_GRID))
k_main = CFG["eval_k"][-1]

cache_path = Path(CFG["cache_path"])
leaderboard_csv_path = cache_path.with_suffix(".leaderboard.csv")

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")
    leaderboard_df = pd.read_csv(leaderboard_csv_path) if leaderboard_csv_path.exists() else None

else:
    frac = float(CFG.get("tune_val_fraction", 1.0))
    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[idx], val_i[idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["tune_epochs"] if CFG["tune_fast"] else CFG["final_epochs"]
    print(
        f"Tuning {len(combos)} trials × {tune_ep} ep "
        f"(val subset: {len(val_u_t):,}/{len(val_u):,})"
    )

    best_hr = -1.0
    best_hp = None
    leaderboard = []

    for ti, (lr, dr, wd, logq_w) in enumerate(combos, 1):
        set_seed(CFG["seed"])
        m = TwoTower(dropout=dr).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)

        status = "ok"
        error_message = ""
        avg_loss = np.nan
        met = None
        hr = -1.0

        try:
            for ep in range(1, tune_ep + 1):
                m.train()
                total_loss = 0.0
                total_n = 0
                for users_batch, items_batch in train_loader:
                    users_batch = users_batch.to(device, non_blocking=True)
                    items_batch = items_batch.to(device, non_blocking=True)
                    user_vecs = m.user_vec(users_batch, ug_t[users_batch])
                    item_vecs = m.item_vec(items_batch, ig_t[items_batch])
                    loss = method_loss(
                        user_vecs=user_vecs,
                        item_vecs=item_vecs,
                        items=items_batch,
                        logq_weight=logq_w,
                    )
                    if not torch.isfinite(loss):
                        raise RuntimeError(f"Non-finite loss: {loss.item()}")
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(m.parameters(), CFG["grad_clip"])
                    opt.step()
                    bs = users_batch.size(0)
                    total_loss += loss.item() * bs
                    total_n += bs
                avg_loss = total_loss / max(total_n, 1)
                print(f"  trial{ti} ep{ep}/{tune_ep} logq_w={logq_w:g} loss={avg_loss:.4f}")

            met = evaluate_full_corpus(
                model=m,
                users=val_u_t,
                true_items=val_i_t,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )
            hr = met["overall"][f"HR@{k_main}"]
            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")

        except RuntimeError as e:
            status = "failed"
            error_message = str(e)
            print(f"  trial {ti} FAILED: {e}")

        row = {
            "trial": ti, "status": status, "error": error_message,
            "lr": lr, "dropout": dr, "weight_decay": wd, "logq_weight": logq_w,
            "tune_epochs": tune_ep, "val_subset_size": len(val_u_t),
            "final_train_loss": avg_loss, f"val_HR@{k_main}": hr,
        }
        if met is not None:
            for split in ("overall", "head", "tail"):
                for key, value in met[split].items():
                    row[f"val_{split}_{key}"] = value
            for key, value in met.get("long_tail", {}).items():
                row[f"val_{key}"] = value
        leaderboard.append(row)

        print(
            f"  trial {ti:3d}/{len(combos)} "
            f"lr={lr:.0e} dr={dr} wd={wd:.0e} logq_w={logq_w:g} "
            f"-> val HR@{k_main}={hr:.2f}%"
        )

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {
                "lr": lr, "dropout": dr, "weight_decay": wd,
                "logq_weight": float(logq_w), f"val_HR@{k_main}": hr,
            }

        del m, opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    leaderboard_df = pd.DataFrame(leaderboard).sort_values(
        f"val_HR@{k_main}", ascending=False, na_position="last"
    )
    leaderboard_df.to_csv(leaderboard_csv_path, index=False)

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial.")

    cache_path.write_text(json.dumps(best_hp, indent=2))
    print(f"\nBest val HR@{k_main}={best_hr:.2f}% -> {best_hp}")
    print(f"Saved best hparams: {cache_path}")
    print(f"Saved leaderboard: {leaderboard_csv_path}")

leaderboard_df.head(10) if leaderboard_df is not None else None

Tuning 6 trials × 3 ep (val subset: 20,194/20,194)
  trial1 ep1/3 logq_w=0.1 loss=6.8132
  trial1 ep2/3 logq_w=0.1 loss=6.6319
  trial1 ep3/3 logq_w=0.1 loss=6.5471
  trial   1/6 lr=1e-03 dr=0.1 wd=0e+00 logq_w=0.1 -> val HR@50=0.42%
  trial2 ep1/3 logq_w=0.5 loss=6.9501
  trial2 ep2/3 logq_w=0.5 loss=6.7958
  trial2 ep3/3 logq_w=0.5 loss=6.7389
  trial   2/6 lr=1e-03 dr=0.1 wd=0e+00 logq_w=0.5 -> val HR@50=0.88%
  trial3 ep1/3 logq_w=1 loss=7.3353
  trial3 ep2/3 logq_w=1 loss=7.1602
  trial3 ep3/3 logq_w=1 loss=7.1183
  trial   3/6 lr=1e-03 dr=0.1 wd=0e+00 logq_w=1 -> val HR@50=0.56%
  trial4 ep1/3 logq_w=2 loss=7.1401
  trial4 ep2/3 logq_w=2 loss=7.1290
  trial4 ep3/3 logq_w=2 loss=7.1223
  trial   4/6 lr=1e-03 dr=0.1 wd=0e+00 logq_w=2 -> val HR@50=0.07%
  trial5 ep1/3 logq_w=5 loss=6.9315
  trial5 ep2/3 logq_w=5 loss=6.9315
  trial5 ep3/3 logq_w=5 loss=6.9315
  trial   5/6 lr=1e-03 dr=0.1 wd=0e+00 logq_w=5 -> val HR@50=0.03%
  trial6 ep1/3 logq_w=10 loss=6.9315
  trial6 ep2/3 logq_w

,trial,status,error,lr,dropout,weight_decay,logq_weight,tune_epochs,val_subset_size,final_train_loss,...,val_CatalogCoverage@10,val_TailCoverage@10,val_AvgPopularity@10,val_TailRatio@10,val_Personalization@10,val_CatalogCoverage@50,val_TailCoverage@50,val_AvgPopularity@50,val_TailRatio@50,val_Personalization@50
1,2,ok,,0.001,0.1,0.0,0.5,3,20194,6.738890,...,3.440095,3.371409,2.460717,96.034466,98.691710,9.091108,9.001107,2.486867,95.340002,97.869695
2,3,ok,,0.001,0.1,0.0,1.0,3,20194,7.118315,...,1.530467,1.489743,2.752986,94.820244,97.553963,3.714458,3.648960,2.747123,95.362088,95.476275
0,1,ok,,0.001,0.1,0.0,0.1,3,20194,6.547087,...,36.276372,36.220788,1.254672,98.983361,99.941531,70.084929,70.055073,1.232903,99.111023,99.819708
3,4,ok,,0.001,0.1,0.0,2.0,3,20194,7.122308,...,2.695602,2.698292,1.130569,100.000000,84.882895,7.204040,7.208316,1.047900,99.995048,78.673564
4,5,ok,,0.001,0.1,0.0,5.0,3,20194,6.931472,...,10.087403,10.082173,0.848804,99.916312,98.523783,25.616954,25.613381,0.847238,99.906606,97.576648
5,6,ok,,0.001,0.1,0.0,10.0,3,20194,6.931472,...,10.087403,10.082173,0.848804,99.916312,98.523783,25.616954,25.613381,0.847238,99.906606,97.576648


## 8. Final training over 5 seeds

In [9]:
all_rows = []
all_test = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"BookCrossing LogQ seed {seed}")
    print("=" * 80)

    set_seed(seed)

    model = TwoTower(dropout=best_hp["dropout"]).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=best_hp["lr"], weight_decay=best_hp["weight_decay"]
    )
    logq_weight = float(best_hp["logq_weight"])

    best_val_hr50 = -1.0
    best_state = None

    for epoch in range(1, CFG["final_epochs"] + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for users_batch, items_batch in train_loader:
            users_batch = users_batch.to(device, non_blocking=True)
            items_batch = items_batch.to(device, non_blocking=True)
            user_vecs = model.user_vec(users_batch, ug_t[users_batch])
            item_vecs = model.item_vec(items_batch, ig_t[items_batch])
            loss = method_loss(
                user_vecs=user_vecs,
                item_vecs=item_vecs,
                items=items_batch,
                logq_weight=logq_weight,
            )
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at seed={seed}, epoch={epoch}: {loss.item()}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()
            bs = users_batch.size(0)
            total_loss += loss.item() * bs
            total_n += bs

        avg_loss = total_loss / max(total_n, 1)

        val_metrics = evaluate_full_corpus(
            model=model,
            users=val_u,
            true_items=val_i,
            known_user_items=known_val,
            head_mask=head_mask,
            ks=CFG["eval_k"],
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )
        hr50 = val_metrics["overall"].get("HR@50", -1.0)

        if hr50 > best_val_hr50:
            best_val_hr50 = hr50
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"seed {seed} epoch {epoch}: loss={avg_loss:.4f} val HR@50={hr50:.4f} [new best]")
        else:
            print(f"seed {seed} epoch {epoch}: loss={avg_loss:.4f} val HR@50={hr50:.4f}")

    print(f"seed {seed} best val HR@50: {best_val_hr50:.4f}")

    assert best_state is not None
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )
    print("TEST METRICS")
    print_metrics(test_metrics)
    all_test.append(test_metrics)

    row = {
        "method": "LogQ",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "logq_weight": best_hp["logq_weight"],
        "best_val_HR@50": best_val_hr50,
    }
    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value
    for key, value in test_metrics.get("long_tail", {}).items():
        row[f"test_{key}"] = value
    for key, value in test_metrics.get("counts", {}).items():
        row[f"test_count_{key}"] = value
    all_rows.append(row)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_rows)
metrics_df

Using best_hp: {'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'logq_weight': 0.5, 'val_HR@50': 0.8764979696939685}

BookCrossing LogQ seed 0
seed 0 epoch 1: loss=6.9501 val HR@50=0.6388 [new best]


KeyboardInterrupt: 

## 9. Compact final summary

In [ ]:
def make_bookcrossing_summary_table(
    metrics_df: pd.DataFrame,
    method_name: str = "LogQ",
    save_path: str | None = "bookcrossing_logq_summary.csv",
) -> pd.DataFrame:
    """One final table: mean ± std over seeds."""
    selected_metrics = [
        "test_overall_HR@10",
        "test_overall_NDCG@10",
        "test_overall_HR@50",
        "test_overall_NDCG@50",
        "test_head_HR@50",
        "test_head_NDCG@50",
        "test_tail_HR@50",
        "test_tail_NDCG@50",
        "test_CatalogCoverage@50",
        "test_TailCoverage@50",
        "test_AvgPopularity@50",
        "test_TailRatio@50",
        "test_Personalization@50",
    ]

    row = {
        "method": method_name,
        "num_seeds": metrics_df["seed"].nunique() if "seed" in metrics_df.columns else len(metrics_df),
        "head_fraction": CFG["head_fraction"],
        "head_share": f"{100 * CFG['head_fraction']:.1f}%",
        "num_head_items": int(head_mask.sum()),
        "num_tail_items": int((~head_mask).sum()),
        "test_head_share": f"{100 * float(head_mask[test_i].mean()):.2f}%",
        "test_tail_share": f"{100 * float((~head_mask[test_i]).mean()):.2f}%",
    }

    for hp_col in ["lr", "dropout", "weight_decay", "logq_weight"]:
        if hp_col in metrics_df.columns:
            vals = metrics_df[hp_col].dropna().unique()
            if len(vals) == 1:
                row[hp_col] = vals[0]

    if "best_val_HR@50" in metrics_df.columns:
        vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

    for metric in selected_metrics:
        if metric not in metrics_df.columns:
            continue
        vals = metrics_df[metric].dropna().to_numpy(dtype=float)
        out_name = metric.replace("test_", "")
        if len(vals) == 0:
            row[out_name] = "nan"
            continue
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        if "AvgPopularity" in out_name:
            row[out_name] = f"{mean:.4f} ± {std:.4f}"
        else:
            row[out_name] = f"{mean:.2f} ± {std:.2f}"

    summary_df = pd.DataFrame([row])
    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")
    return summary_df


final_summary_df = make_bookcrossing_summary_table(
    metrics_df=metrics_df,
    method_name="LogQ",
    save_path="bookcrossing_logq_summary.csv",
)
final_summary_df